```python
import threading
from queue import Queue
from concurrent.futures import ThreadPoolExecutor

class SeriesCoordinator:
    def __init__(self, loader_registry, cleaner_registry, storage):
        self.loader_registry = loader_registry
        self.cleaner_registry = cleaner_registry
        self.storage = storage
        self.cleaning_queue = Queue()

    def handle_request(self, series_list):
        grouped = self._group_by_source(series_list)

        # Start cleaner thread
        threading.Thread(target=self._clean_worker, daemon=True).start()

        with ThreadPoolExecutor(max_workers=4) as executor:
            for source, series_ids in grouped.items():
                for series_id in series_ids:
                    executor.submit(self._download_and_enqueue, source, series_id)

        self.cleaning_queue.join()  # Wait until all cleaning tasks are done

    def _group_by_source(self, series_list):
        grouped = {}
        for series in series_list:
            source = self._resolve_source(series)
            grouped.setdefault(source, []).append(series)
        return grouped

    def _download_and_enqueue(self, source, series_id):
        loader = self.loader_registry[source]
        data = loader.load(series_id)  # raw JSON, etc.

        self.cleaning_queue.put((source, series_id, data))

    def _clean_worker(self):
        while True:
            source, series_id, raw_data = self.cleaning_queue.get()
            try:
                cleaner = self.cleaner_registry[source]
                cleaned = cleaner.clean(series_id, raw_data)
                self.storage.save_raw(series_id, cleaned)
            finally:
                self.cleaning_queue.task_done()

    def _resolve_source(self, series_id):
        # Could be a lookup from catalog
        if series_id.startswith("SGS_"):
            return "SGS"
        elif series_id.startswith("IBGE_"):
            return "IBGE"
        else:
            raise ValueError(f"Unknown source for series: {series_id}")
